In [1]:
import numpy as np
import gym
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from envs import TFAugmentedGridWorldEnv2

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
n_thought_acts = 3
n_thought_states = 10
d_model = 16
seed = 42

In [3]:
env = TFAugmentedGridWorldEnv2(
    n_thought_states=n_thought_states,
    n_thought_acts=n_thought_acts,
    n_goals=2,
    deterministic_start=True,
    d_model=d_model,
    seed=seed,
)

In [4]:
obs = env.reset()

In [5]:
obs

{'letter': np.int64(1),
 'position': [3, 3],
 'thought': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])}

In [6]:
next_obss = []
for act in range(5 + n_thought_acts):
    _ = env.reset()
    next_obs, rew, done, _ = env.step(act)
    next_obss.append(
        np.concatenate([np.array([v], dtype=np.float32) if k == "letter" else np.array(v, dtype=np.float32) for k, v in next_obs.items()], axis=0)
    )

/tmp/ipykernel_1065480/2551814597.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  np.concatenate([np.array([v], dtype=np.float32) if k == "letter" else np.array(v, dtype=np.float32) for k, v in next_obs.items()], axis=0)


In [7]:
next_obss

[array([2., 3., 3., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([1., 2., 3., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([1., 4., 3., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([1., 3., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([2., 3., 4., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([ 1.        ,  3.        ,  3.        ,  0.46524566, -0.20657514,
        -0.3988726 , -0.18239205,  0.09476374,  0.44416705, -0.09531733,
         0.04156468, -0.27183622, -0.18161179,  0.4390949 , -0.4735986 ,
         0.3205355 ,  0.2652688 ,  0.4504122 , -0.24278216], dtype=float32),
 array([ 1.        ,  3.        ,  3.        , -0.27866328,  0.02849502,
        -0.40749925, -0.32763   , -0.23626818,  0.2109445 ,  0.1601114 ,
        -0.36

In [8]:
np.linalg.matrix_rank(next_obss)

np.int64(6)

In [9]:
_ = env.reset()
for _ in range(10):
    act = np.random.randint(5, 8)
    next_obs, rew, done, _ = env.step(act)
    print(act, next_obs)

5 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
5 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
7 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
5 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
7 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.16

In [10]:
env.thought_state_embedding.weight @ env.thought_state_embedding.weight.T

tensor([[ 0.9853,  0.1676,  0.1450,  0.1315, -0.1479,  0.1126, -0.1901,  0.5625,
          0.1934,  0.3266],
        [ 0.1676,  1.1225, -0.4512, -0.3640, -0.3535, -0.2992,  0.2221,  0.2527,
         -0.2748,  0.4015],
        [ 0.1450, -0.4512,  1.4548, -0.3283, -0.2635,  0.6973,  0.1870,  0.0210,
          0.3630,  0.5025],
        [ 0.1315, -0.3640, -0.3283,  1.1275, -0.0815, -0.4468, -0.0888, -0.2139,
         -0.0447, -0.2639],
        [-0.1479, -0.3535, -0.2635, -0.0815,  0.8634, -0.0472, -0.1117,  0.0556,
          0.0932, -0.5424],
        [ 0.1126, -0.2992,  0.6973, -0.4468, -0.0472,  1.5890,  0.8646, -0.0985,
          0.7088,  0.0106],
        [-0.1901,  0.2221,  0.1870, -0.0888, -0.1117,  0.8646,  1.8710, -0.2938,
          0.7573,  0.0505],
        [ 0.5625,  0.2527,  0.0210, -0.2139,  0.0556, -0.0985, -0.2938,  1.3132,
          0.1067,  0.0509],
        [ 0.1934, -0.2748,  0.3630, -0.0447,  0.0932,  0.7088,  0.7573,  0.1067,
          1.0633,  0.3197],
        [ 0.3266,  

In [11]:
env.thought_state_embedding.weight.shape

torch.Size([10, 16])